# Graph Analysis – Unified: Advanced Analytics

This notebook performs a comprehensive structural analysis of the Custom-WikiCS graph:

1. **Removal of Isolated Nodes**: Nodes with degree 0 are removed before analysis.
2. **Connected Component Analysis**: WCC/SCC counts, sizes, and distributions.
3. **Centrality Measures**: Degree, Betweenness, Closeness, PageRank, Eigenvector.
4. **Community Detection**: Louvain and Infomap algorithms with comparison.
5. **Community Similarity Analysis**: Intra-community vs inter-community link similarity.
6. **Degree vs. Similarity Analysis**: Node degree ranges vs average neighbor similarity.

All heavy computations are cached in `./cache/graph-analysis-4/` for faster subsequent runs.

## 1. Setup and Data Loading

In [ ]:
import json, os, pickle
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity

CACHE_DIR = './cache/graph-analysis-4'
IMAGE_DIR = './graph-analysis-4-img'
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(IMAGE_DIR, exist_ok=True)

plt.style.use('bmh')
sns.set_palette("viridis")
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['font.size'] = 12

In [ ]:
def compute_or_load(name, compute_fn, force=False):
    path = os.path.join(CACHE_DIR, f'unified_{name}.pkl')
    if os.path.exists(path) and not force:
        print(f'  [cache] {name} loaded from cache.')
        with open(path, 'rb') as f:
            return pickle.load(f)
    print(f'  [comp]  Computing {name}…')
    result = compute_fn()
    with open(path, 'wb') as f:
        pickle.dump(result, f)
    print(f'  [save]  {name} computed and saved.')
    return result

In [ ]:
# Load node and edge data
with open('./data/data-embeddings.json', 'r') as f:
    raw_data = json.load(f)
print(f"Total node data: {len(raw_data['nodes'])}")

# Load embeddings
df_embeddings = pd.read_parquet('./cap-embeddings/BAAI_bge-m3/master_embeddings.parquet')
embedding_dict = dict(zip(df_embeddings['id'].astype(int), df_embeddings['embedding']))
print(f"Nodes with embedding data: {len(df_embeddings)}")

## 2. Removal of Isolated Nodes

Nodes with degree 0 (no incoming or outgoing edges) are removed from the graph.

In [ ]:
# Build full graph first to identify isolates
node_ids = {n['id'] for n in raw_data['nodes']}
G_full = nx.DiGraph()
for n in raw_data['nodes']:
    G_full.add_node(n['id'], title=n['title'], label=n.get('label', 'Unknown'))
for n in raw_data['nodes']:
    src = n['id']
    for tgt in n.get('outlinks', []):
        if tgt in node_ids:
            G_full.add_edge(src, tgt)

# Identify isolated nodes
isolated_nodes = list(nx.isolates(G_full))
print(f"Number of isolated nodes removed: {len(isolated_nodes)}")

# Create cleaned graph (removes isolates)
G = G_full.copy()
G.remove_nodes_from(isolated_nodes)
G_und = G.to_undirected()

print(f"New node count: {G.number_of_nodes()}")
print(f"New edge count: {G.number_of_edges()}")
print(f"New undirected edge count: {G_und.number_of_edges()}")

## 2.5. Degree Distribution and Assortativity Analysis

This section analyzes:
1. **Degree Distribution**: Total, In, and Out-degree histograms
2. **Log-Log Degree Distribution**: Power-law check
3. **Topic-Link Assortativity**: Cosine, Pearson, and Euclidean similarity distributions for linked vs random node pairs
4. **Local Transitivity (Clustering Coefficient)**: Distribution and statistics
5. **Degree vs. Similarity**: Scatter plots comparing node degree with embedding similarity metrics
6. **Transitivity vs. Similarity**: Scatter plots comparing local transitivity with cosine similarity


In [ ]:
# Degree Distribution Analysis
degrees = [G.degree(n) for n in G.nodes()]
in_degrees = [G.in_degree(n) for n in G.nodes()]
out_degrees = [G.out_degree(n) for n in G.nodes()]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(degrees, bins=50, color='#2196F3', edgecolor='black', alpha=0.85)
axes[0].set_title('Total Degree Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Degree', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].axvline(np.mean(degrees), color='red', linestyle='--', label=f'Mean: {np.mean(degrees):.2f}')
axes[0].legend(fontsize=10)

axes[1].hist(in_degrees, bins=50, color='#4CAF50', edgecolor='black', alpha=0.85)
axes[1].set_title('In-Degree Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('In-Degree', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].axvline(np.mean(in_degrees), color='red', linestyle='--', label=f'Mean: {np.mean(in_degrees):.2f}')
axes[1].legend(fontsize=10)

axes[2].hist(out_degrees, bins=50, color='#FF9800', edgecolor='black', alpha=0.85)
axes[2].set_title('Out-Degree Distribution', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Out-Degree', fontsize=12)
axes[2].set_ylabel('Frequency', fontsize=12)
axes[2].axvline(np.mean(out_degrees), color='red', linestyle='--', label=f'Mean: {np.mean(out_degrees):.2f}')
axes[2].legend(fontsize=10)

plt.suptitle('Degree Distribution Histograms', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{IMAGE_DIR}/degree-distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nDegree Statistics:")
print(f"  Total Degree  -> Min: {min(degrees)}, Max: {max(degrees)}, Mean: {np.mean(degrees):.2f}, Median: {np.median(degrees):.0f}")
print(f"  In-Degree     -> Min: {min(in_degrees)}, Max: {max(in_degrees)}, Mean: {np.mean(in_degrees):.2f}, Median: {np.median(in_degrees):.0f}")
print(f"  Out-Degree    -> Min: {min(out_degrees)}, Max: {max(out_degrees)}, Mean: {np.mean(out_degrees):.2f}, Median: {np.median(out_degrees):.0f}")

In [ ]:
# Log-Log Scale Degree Distribution (Power-Law Check)
degree_count = Counter(degrees)
deg, cnt = zip(*sorted(degree_count.items()))

plt.figure(figsize=(8, 6))
plt.loglog(deg, cnt, 'o', markersize=5, color='#E91E63', alpha=0.7)
plt.title('Degree Distribution (Log-Log Scale)', fontsize=14, fontweight='bold')
plt.xlabel('Degree (log)', fontsize=12)
plt.ylabel('Frequency (log)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{IMAGE_DIR}/degree-log-log.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Degree Assortativity Coefficient
# Positive value: high-degree nodes tend to connect to high-degree nodes (assortative)
# Negative value: high-degree nodes tend to connect to low-degree nodes (disassortative)

def compute_assortativity():
    return nx.degree_assortativity_coefficient(G)

r = compute_or_load('degree_assortativity', compute_assortativity)
print(f"Degree Assortativity Coefficient: {r:.6f}")

if r > 0:
    print("Network shows ASSORTATIVE behavior (similar degree nodes connect)")
else:
    print("Network shows DISASSORTATIVE behavior (different degree nodes connect)")


In [ ]:
# Compute edge similarity metrics using embeddings
from scipy.spatial.distance import cosine, euclidean
from scipy.stats import pearsonr

def compute_edge_metrics():
    cosine_sims = []
    pearson_corrs = []
    euclidean_dists = []
    skipped = 0
    for u, v in G.edges():
        if u in embedding_dict and v in embedding_dict:
            emb_u = embedding_dict[u]
            emb_v = embedding_dict[v]
            cos_sim = 1 - cosine(emb_u, emb_v)
            cosine_sims.append(cos_sim)
            corr, _ = pearsonr(emb_u, emb_v)
            pearson_corrs.append(corr)
            euc_dist = euclidean(emb_u, emb_v)
            euclidean_dists.append(euc_dist)
        else:
            skipped += 1
    return cosine_sims, pearson_corrs, euclidean_dists, skipped

cosine_sims, pearson_corrs, euclidean_dists, skipped = compute_or_load(
    'edge_similarity_metrics', compute_edge_metrics)

print(f"Edges with computed metrics: {len(cosine_sims)}")
print(f"Edges skipped (missing embedding): {skipped}")


In [ ]:
# Compute random (non-linked) node pair metrics for baseline comparison
np.random.seed(42)
node_list = [n for n in G.nodes() if n in embedding_dict]
n_random = min(len(cosine_sims), 50000)

def compute_random_metrics():
    random_cosine = []
    random_pearson = []
    random_euclid = []
    for _ in range(n_random):
        i, j = np.random.choice(len(node_list), 2, replace=False)
        emb_u = embedding_dict[node_list[i]]
        emb_v = embedding_dict[node_list[j]]
        random_cosine.append(1 - cosine(emb_u, emb_v))
        corr, _ = pearsonr(emb_u, emb_v)
        random_pearson.append(corr)
        random_euclid.append(euclidean(emb_u, emb_v))
    return random_cosine, random_pearson, random_euclid

random_cosine, random_pearson, random_euclid = compute_or_load(
    'random_pair_metrics', compute_random_metrics)

print(f"Random pairs computed: {n_random}")


In [ ]:
# Cosine Similarity: Linked vs Random Node Pairs
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(cosine_sims, bins=60, alpha=0.7, color='#2196F3', edgecolor='black', label=f'Linked Pairs (mean: {np.mean(cosine_sims):.4f})', density=True)
ax.hist(random_cosine, bins=60, alpha=0.5, color='#FF5722', edgecolor='black', label=f'Random Pairs (mean: {np.mean(random_cosine):.4f})', density=True)
ax.set_title('Cosine Similarity: Linked vs Random Node Pairs', fontsize=14, fontweight='bold')
ax.set_xlabel('Cosine Similarity', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{IMAGE_DIR}/cosine-similarity-assortativity.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Linked Pairs - Cosine Sim Mean: {np.mean(cosine_sims):.4f}, Std: {np.std(cosine_sims):.4f}")
print(f"Random Pairs - Cosine Sim Mean: {np.mean(random_cosine):.4f}, Std: {np.std(random_cosine):.4f}")

In [ ]:
# Pearson Correlation: Linked vs Random Node Pairs
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(pearson_corrs, bins=60, alpha=0.7, color='#4CAF50', edgecolor='black', label=f'Linked Pairs (mean: {np.mean(pearson_corrs):.4f})', density=True)
ax.hist(random_pearson, bins=60, alpha=0.5, color='#FF5722', edgecolor='black', label=f'Random Pairs (mean: {np.mean(random_pearson):.4f})', density=True)
ax.set_title('Pearson Correlation: Linked vs Random Node Pairs', fontsize=14, fontweight='bold')
ax.set_xlabel('Pearson Correlation', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{IMAGE_DIR}/pearson-correlation-assortativity.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Linked Pairs - Pearson Corr Mean: {np.mean(pearson_corrs):.4f}, Std: {np.std(pearson_corrs):.4f}")
print(f"Random Pairs - Pearson Corr Mean: {np.mean(random_pearson):.4f}, Std: {np.std(random_pearson):.4f}")

In [ ]:
# Euclidean Distance: Linked vs Random Node Pairs
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(euclidean_dists, bins=60, alpha=0.7, color='#9C27B0', edgecolor='black', label=f'Linked Pairs (mean: {np.mean(euclidean_dists):.4f})', density=True)
ax.hist(random_euclid, bins=60, alpha=0.5, color='#FF5722', edgecolor='black', label=f'Random Pairs (mean: {np.mean(random_euclid):.4f})', density=True)
ax.set_title('Euclidean Distance: Linked vs Random Node Pairs', fontsize=14, fontweight='bold')
ax.set_xlabel('Euclidean Distance', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{IMAGE_DIR}/euclidean-distance-assortativity.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Linked Pairs - Euclidean Dist Mean: {np.mean(euclidean_dists):.4f}, Std: {np.std(euclidean_dists):.4f}")
print(f"Random Pairs - Euclidean Dist Mean: {np.mean(random_euclid):.4f}, Std: {np.std(random_euclid):.4f}")

In [ ]:
# Topic-Link Assortativity Summary Table
summary_data = {
    'Metric': ['Cosine Similarity', 'Pearson Correlation', 'Euclidean Distance'],
    'Linked Mean': [np.mean(cosine_sims), np.mean(pearson_corrs), np.mean(euclidean_dists)],
    'Linked Std': [np.std(cosine_sims), np.std(pearson_corrs), np.std(euclidean_dists)],
    'Random Mean': [np.mean(random_cosine), np.mean(random_pearson), np.mean(random_euclid)],
    'Random Std': [np.std(random_cosine), np.std(random_pearson), np.std(random_euclid)],
}
summary_df = pd.DataFrame(summary_data)
print("\n=== Topic-Link Assortativity Summary ===")
print(summary_df.to_string(index=False))

In [ ]:
# Local Transitivity (Clustering Coefficient)

def compute_clustering():
    clustering_coeffs = nx.clustering(G_und)
    return list(clustering_coeffs.values())

cc_values = compute_or_load('clustering_coeffs', compute_clustering)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(cc_values, bins=50, color='#00BCD4', edgecolor='black', alpha=0.85)
axes[0].set_title('Local Transitivity (Clustering Coefficient) Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Clustering Coefficient', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].axvline(np.mean(cc_values), color='red', linestyle='--', label=f'Mean: {np.mean(cc_values):.4f}')
axes[0].legend(fontsize=10)

cc_nonzero = [c for c in cc_values if c > 0]
axes[1].hist(cc_nonzero, bins=50, color='#FF9800', edgecolor='black', alpha=0.85)
axes[1].set_title('Local Transitivity (CC > 0 only)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Clustering Coefficient', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].axvline(np.mean(cc_nonzero), color='red', linestyle='--', label=f'Mean (>0): {np.mean(cc_nonzero):.4f}')
axes[1].legend(fontsize=10)

plt.suptitle('Local Transitivity Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{IMAGE_DIR}/local-transitivity-distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nLocal Transitivity Statistics:")
print(f"  Mean: {np.mean(cc_values):.4f}")
print(f"  Median: {np.median(cc_values):.4f}")
print(f"  Min: {min(cc_values):.4f}, Max: {max(cc_values):.4f}")
print(f"  Std: {np.std(cc_values):.4f}")
print(f"  Global Transitivity (graph-wide): {nx.transitivity(G_und):.4f}")


In [ ]:
# Degree vs Similarity Metrics - Compute

def compute_degree_vs_similarity():
    src_degrees = []
    tgt_degrees = []
    edge_cosine_arr = []
    edge_pearson_arr = []
    edge_euclidean_arr = []
    for u, v in G.edges():
        if u in embedding_dict and v in embedding_dict:
            src_degrees.append(G.degree(u))
            tgt_degrees.append(G.degree(v))
            edge_cosine_arr.append(1 - cosine(embedding_dict[u], embedding_dict[v]))
            corr, _ = pearsonr(embedding_dict[u], embedding_dict[v])
            edge_pearson_arr.append(corr)
            edge_euclidean_arr.append(euclidean(embedding_dict[u], embedding_dict[v]))
    return (np.array(src_degrees), np.array(tgt_degrees), np.array(edge_cosine_arr),
            np.array(edge_pearson_arr), np.array(edge_euclidean_arr))

src_degrees, tgt_degrees, edge_cosine_arr, edge_pearson_arr, edge_euclidean_arr = \
    compute_or_load('degree_vs_similarity_data', compute_degree_vs_similarity)

print(f"Total edges (with embeddings): {len(src_degrees)}")


In [ ]:
# Degree vs Similarity Metrics - Scatter Plots
metrics = [
    ('Cosine Similarity', edge_cosine_arr, '#2196F3'),
    ('Euclidean Distance', edge_euclidean_arr, '#9C27B0'),
    ('Pearson Correlation', edge_pearson_arr, '#4CAF50'),
]

fig, axes = plt.subplots(3, 2, figsize=(18, 18))

for row, (name, vals, color) in enumerate(metrics):
    for col, (deg_arr, role) in enumerate([(src_degrees, 'Source'), (tgt_degrees, 'Target')]):
        ax = axes[row][col]
        ax.scatter(deg_arr, vals, s=1, alpha=0.15, color=color)
        ax.set_xlabel(f'{role} Node Degree', fontsize=12)
        ax.set_ylabel(name, fontsize=12)
        ax.set_title(f'{name} vs {role} Node Degree', fontsize=13, fontweight='bold')
        ax.grid(True, alpha=0.3)

plt.suptitle('Similarity Metrics vs Node Degree (6 Comparisons)', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{IMAGE_DIR}/degree-vs-similarity-metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Local Transitivity vs Cosine Similarity - Compute

def compute_transitivity_vs_similarity():
    cc_dict = nx.clustering(G_und)
    src_cc = []
    tgt_cc = []
    cc_cosine_arr = []
    for u, v in G.edges():
        if u in embedding_dict and v in embedding_dict:
            src_cc.append(cc_dict[u])
            tgt_cc.append(cc_dict[v])
            cc_cosine_arr.append(1 - cosine(embedding_dict[u], embedding_dict[v]))
    return np.array(src_cc), np.array(tgt_cc), np.array(cc_cosine_arr)

src_cc, tgt_cc, cc_cosine_arr = \
    compute_or_load('transitivity_vs_similarity_data', compute_transitivity_vs_similarity)

print(f"Total edges (CC computed): {len(src_cc)}")


In [ ]:
# Local Transitivity vs Cosine Similarity - Scatter Plots
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for col, (cc_arr, role) in enumerate([(src_cc, 'Source'), (tgt_cc, 'Target')]):
    ax = axes[col]
    ax.scatter(cc_arr, cc_cosine_arr, s=1, alpha=0.15, color='#E91E63')
    ax.set_xlabel(f'{role} Node Local Transitivity (CC)', fontsize=12)
    ax.set_ylabel('Cosine Similarity', fontsize=12)
    ax.set_title(f'Cosine Similarity vs {role} Node Local Transitivity', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('Cosine Similarity vs Node Local Transitivity (2 Comparisons)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{IMAGE_DIR}/transitivity-vs-similarity.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Connected Component Analysis

In [ ]:
# Directed graph: weakly and strongly connected components
n_wcc = nx.number_weakly_connected_components(G)
n_scc = nx.number_strongly_connected_components(G)

# Largest components
largest_wcc = max(nx.weakly_connected_components(G), key=len)
largest_scc = max(nx.strongly_connected_components(G), key=len)

print(f"Weakly Connected Components count:  {n_wcc}")
print(f"Strongly Connected Components count: {n_scc}")
print(f"Largest WCC size: {len(largest_wcc)} nodes ({100*len(largest_wcc)/G.number_of_nodes():.1f}%)")
print(f"Largest SCC size: {len(largest_scc)} nodes ({100*len(largest_scc)/G.number_of_nodes():.1f}%)")

In [ ]:
# Component size distributions
wcc_sizes = [len(c) for c in nx.weakly_connected_components(G)]
scc_sizes = [len(c) for c in nx.strongly_connected_components(G)]

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(wcc_sizes, bins=30, kde=True, ax=ax[0], color='skyblue')
ax[0].set_title('Weakly Connected Components Size Distribution')
ax[0].set_yscale('log')
ax[0].set_xlabel('Component Size')
ax[0].set_ylabel('Frequency (Log)')

sns.histplot(scc_sizes, bins=30, kde=True, ax=ax[1], color='salmon')
ax[1].set_title('Strongly Connected Components Size Distribution')
ax[1].set_yscale('log')
ax[1].set_xlabel('Component Size')
ax[1].set_ylabel('Frequency (Log)')

plt.tight_layout()
plt.savefig(f'{IMAGE_DIR}/component-distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Centrality Measures

Five centrality metrics are computed:
- **Degree Centrality** (directed graph)
- **Betweenness Centrality** (undirected graph)
- **Closeness Centrality** (undirected graph)
- **PageRank** (directed graph)
- **Eigenvector Centrality** (undirected graph)

In [ ]:
# Compute all centrality metrics with caching
degree_cent = compute_or_load('degree_centrality', lambda: nx.degree_centrality(G))
betweenness_cent = compute_or_load('betweenness_centrality', lambda: nx.betweenness_centrality(G_und))
closeness_cent = compute_or_load('closeness_centrality', lambda: nx.closeness_centrality(G_und))
pagerank_cent = compute_or_load('pagerank', lambda: nx.pagerank(G))
eigenvector_cent = compute_or_load('eigenvector_centrality', lambda: nx.eigenvector_centrality(G_und, max_iter=1000))

In [ ]:
def get_stats(name, d):
    vals = list(d.values())
    return {
        'Metric': name,
        'Min': np.min(vals),
        'Max': np.max(vals),
        'Mean': np.mean(vals),
        'Median': np.median(vals),
        'Std': np.std(vals)
    }

stats_df = pd.DataFrame([
    get_stats('Degree', degree_cent),
    get_stats('Betweenness', betweenness_cent),
    get_stats('Closeness', closeness_cent),
    get_stats('PageRank', pagerank_cent),
    get_stats('Eigenvector', eigenvector_cent)
])
print("Centrality Statistics:")
print(stats_df.to_string(index=False))

In [ ]:
# Centrality distribution histograms
centralities = [
    ('Degree Centrality', degree_cent, '#2196F3'),
    ('Betweenness Centrality', betweenness_cent, '#4CAF50'),
    ('Closeness Centrality', closeness_cent, '#FF9800'),
    ('PageRank', pagerank_cent, '#9C27B0'),
    ('Eigenvector Centrality', eigenvector_cent, '#00BCD4'),
]

fig, axes = plt.subplots(3, 2, figsize=(16, 18))

for idx, (name, cent_dict, color) in enumerate(centralities):
    ax = axes[idx // 2][idx % 2]
    vals = list(cent_dict.values())
    ax.hist(vals, bins=80, color=color, edgecolor='black', alpha=0.85)
    ax.set_title(f'{name} Distribution', fontsize=13, fontweight='bold')
    ax.set_xlabel(name, fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.axvline(np.mean(vals), color='red', linestyle='--', label=f'Mean: {np.mean(vals):.6f}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

axes[2][1].axis('off')  # Hide unused subplot
plt.suptitle('Centrality Distribution Histograms', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{IMAGE_DIR}/centrality-histograms.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top-10 nodes for each centrality metric
for name, cent_dict in [('Degree', degree_cent), ('Betweenness', betweenness_cent),
                         ('Closeness', closeness_cent), ('PageRank', pagerank_cent),
                         ('Eigenvector', eigenvector_cent)]:
    top10 = sorted(cent_dict.items(), key=lambda x: x[1], reverse=True)[:10]
    print(f'\n=== Top-10 {name} Centrality ===')
    for rank, (nid, val) in enumerate(top10, 1):
        title = G.nodes[nid].get('title', nid)
        print(f'  {rank:2d}. {title}  ({val:.6f})')

## 5. Community Detection

Two community detection algorithms are applied:
- **Louvain**: Maximizes modularity
- **Infomap**: Minimizes map equation (random walk approach)

For each algorithm, we compute:
- Modularity score
- Number of communities
- Community size statistics (min, max, mean, median, std)
- Size distribution histogram

### 5.1 Louvain Algorithm

In [ ]:
import community as community_louvain

def run_louvain():
    partition = community_louvain.best_partition(G_und)
    mod = community_louvain.modularity(partition, G_und)
    return partition, mod

louvain_result = compute_or_load('louvain_partition', run_louvain)
louvain_partition, louvain_modularity = louvain_result

print(f'Louvain Modularity: {louvain_modularity:.6f}')

In [ ]:
def community_stats(partition, algo_name):
    comm_sizes = Counter(partition.values())
    sizes = np.array(list(comm_sizes.values()))
    n_communities = len(comm_sizes)

    print(f'\n=== {algo_name} Community Statistics ===')
    print(f'  Community count : {n_communities}')
    print(f'  Min size        : {sizes.min()}')
    print(f'  Max size        : {sizes.max()}')
    print(f'  Mean size       : {sizes.mean():.2f}')
    print(f'  Median size     : {np.median(sizes):.1f}')
    print(f'  Std deviation   : {sizes.std():.2f}')

    # Histogram
    plt.figure(figsize=(10, 5))
    plt.hist(sizes, bins=max(20, n_communities // 2), color='#00BCD4', edgecolor='black', alpha=0.85)
    plt.title(f'{algo_name} – Community Size Distribution', fontsize=14, fontweight='bold')
    plt.xlabel('Community Size (node count)', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    plt.axvline(sizes.mean(), color='red', linestyle='--', label=f'Mean: {sizes.mean():.1f}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{IMAGE_DIR}/community-size-distribution-{algo_name.lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()

    return {'n_communities': n_communities, 'sizes': sizes}

louvain_stats = community_stats(louvain_partition, 'Louvain')

### 5.2 Infomap Algorithm

In [ ]:
import infomap

def run_infomap():
    im = infomap.Infomap(seed=42)
    node_to_idx = {n: i for i, n in enumerate(G_und.nodes())}
    idx_to_node = {i: n for n, i in node_to_idx.items()}
    for u, v in G_und.edges():
        im.add_link(node_to_idx[u], node_to_idx[v])
    im.run()

    # Build partition
    partition = {}
    for node_id, module_id in im.modules:
        if node_id in idx_to_node:
            partition[idx_to_node[node_id]] = module_id

    # Handle nodes not assigned by Infomap
    all_nodes = set(G_und.nodes())
    assigned = set(partition.keys())
    missing = all_nodes - assigned
    if missing:
        next_id = max(partition.values()) + 1 if partition else 0
        for node in missing:
            partition[node] = next_id
            next_id += 1
        print(f'  [warn] {len(missing)} nodes not assigned by Infomap, added as singleton communities.')

    # Calculate modularity
    communities_sets = {}
    for nid, comm_id in partition.items():
        communities_sets.setdefault(comm_id, set()).add(nid)
    comm_list = list(communities_sets.values())
    mod = nx.community.modularity(G_und, comm_list)
    return partition, mod

infomap_result = compute_or_load('infomap_partition', run_infomap)
infomap_partition, infomap_modularity = infomap_result

print(f'Infomap Modularity: {infomap_modularity:.6f}')

In [ ]:
infomap_stats = community_stats(infomap_partition, 'Infomap')

### 5.3 Louvain vs Infomap Comparison

In [ ]:
comp = pd.DataFrame([
    {
        'Algorithm': 'Louvain',
        'Modularity': round(louvain_modularity, 6),
        'Communities': louvain_stats['n_communities'],
        'Min Size': louvain_stats['sizes'].min(),
        'Max Size': louvain_stats['sizes'].max(),
        'Mean Size': round(louvain_stats['sizes'].mean(), 2),
        'Median Size': round(float(np.median(louvain_stats['sizes'])), 1),
        'Std': round(louvain_stats['sizes'].std(), 2),
    },
    {
        'Algorithm': 'Infomap',
        'Modularity': round(infomap_modularity, 6),
        'Communities': infomap_stats['n_communities'],
        'Min Size': infomap_stats['sizes'].min(),
        'Max Size': infomap_stats['sizes'].max(),
        'Mean Size': round(infomap_stats['sizes'].mean(), 2),
        'Median Size': round(float(np.median(infomap_stats['sizes'])), 1),
        'Std': round(infomap_stats['sizes'].std(), 2),
    },
])

print('\n=== Louvain vs Infomap Comparison ===')
print(comp.to_string(index=False))

## 6. Community Similarity Analysis

This analysis compares the cosine similarity of node embeddings for:
- **Intra-community links**: Undirected edges where both endpoints belong to the same community
- **Inter-community links**: Undirected edges where endpoints belong to different communities

Higher intra-community similarity suggests that community detection successfully grouped semantically related nodes.

In [ ]:
# Use Louvain partition for community assignments (using undirected graph)
nx.set_node_attributes(G_und, louvain_partition, 'community')

intra_sims = []
inter_sims = []

for u, v in G_und.edges():
    if u in embedding_dict and v in embedding_dict:
        sim = cosine_similarity(embedding_dict[u].reshape(1, -1), embedding_dict[v].reshape(1, -1))[0][0]
        if louvain_partition.get(u) == louvain_partition.get(v):
            intra_sims.append(sim)
        else:
            inter_sims.append(sim)

print(f"Average similarity within communities (intra): {np.mean(intra_sims):.4f}")
print(f"Average similarity between communities (inter): {np.mean(inter_sims):.4f}")

# ============================================================
# Infomap Similarity Analysis
# ============================================================
intra_sims_infomap = []
inter_sims_infomap = []

for u, v in G_und.edges():
    if u in embedding_dict and v in embedding_dict:
        sim = cosine_similarity(embedding_dict[u].reshape(1, -1), embedding_dict[v].reshape(1, -1))[0][0]
        if infomap_partition.get(u) == infomap_partition.get(v):
            intra_sims_infomap.append(sim)
        else:
            inter_sims_infomap.append(sim)

print(f"\nInfomap Average similarity within communities (intra): {np.mean(intra_sims_infomap):.4f}")
print(f"Infomap Average similarity between communities (inter): {np.mean(inter_sims_infomap):.4f}")

# ============================================================
# Combined Similarity Table
# ============================================================
sim_table = pd.DataFrame({
    'Algorithm': ['Louvain', 'Infomap'],
    'Intra-Community Similarity': [np.mean(intra_sims), np.mean(intra_sims_infomap)],
    'Inter-Community Similarity': [np.mean(inter_sims), np.mean(inter_sims_infomap)]
})
print("\n=== Average Cosine Similarity Comparison ===")
print(sim_table.to_string(index=False))

# ============================================================
# Box Plots: Similarity Distributions (Raw Data)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Louvain box plot
axes[0].boxplot([intra_sims, inter_sims], labels=['Intra-Community', 'Inter-Community'])
axes[0].set_title('Louvain: Similarity Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Cosine Similarity')
axes[0].grid(True, alpha=0.3)

# Infomap box plot
axes[1].boxplot([intra_sims_infomap, inter_sims_infomap], labels=['Intra-Community', 'Inter-Community'])
axes[1].set_title('Infomap: Similarity Distribution', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Cosine Similarity')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Community Similarity Distributions (Raw Values)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{IMAGE_DIR}/similarity-distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Degree vs. Similarity Box Plot

This analysis examines how node degree (undirected) correlates with average neighbor similarity.
Nodes are grouped into degree ranges, and the average cosine similarity of their neighbors is plotted.

In [ ]:
node_similarities = []
for u in G_und.nodes():
    neighbors = list(G_und.neighbors(u))
    if not neighbors or u not in embedding_dict:
        continue
    
    sims = []
    u_emb = embedding_dict[u].reshape(1, -1)
    for v in neighbors:
        if v in embedding_dict:
            sims.append(cosine_similarity(u_emb, embedding_dict[v].reshape(1, -1))[0][0])
    
    if sims:
        node_similarities.append({
            'node': u,
            'degree': G_und.degree(u),
            'avg_sim': np.mean(sims)
        })

sim_df = pd.DataFrame(node_similarities)

bins = [0, 3, 10, 30, 100, 300, 500, 1000, 1500, 2000, 2500, 3000, 3500]
labels = ['0-3', '3-10', '10-30', '30-100', '100-300', '300-500', '500-1000', '1000-1500', '1500-2000', '2000-2500', '2500-3000', '3000-3500']
sim_df['degree_bin'] = pd.cut(sim_df['degree'], bins=bins, labels=labels)

plt.figure(figsize=(16, 8))
sns.boxplot(x='degree_bin', y='avg_sim', data=sim_df, palette='Spectral', hue='degree_bin', legend=False)
plt.title('Node Degree vs. Average Neighbor Similarity')
plt.xlabel('Degree Range')
plt.ylabel('Average Cosine Similarity')
plt.xticks(rotation=45)
plt.savefig(f'{IMAGE_DIR}/degree-vs-similarity.png', dpi=150, bbox_inches='tight')
plt.show()